# Frozen Qwen3-Embedding-0.6B

**Question (Further Work item 3):** how does a model trained *to produce embeddings* compare with RoBERTa and ModernBERT, which were only pretrained on masked-word prediction? Qwen3-Embedding (June 2025) starts from the Qwen3 language model and is then trained with contrastive learning, so that one vector summarizes a text. Its 8B version topped the multilingual MTEB leaderboard when it came out; this is the smallest version: 28 layers, 0.6B parameters (RoBERTa: 125M), 1,024-d output.

Same data, split and classifier settings as `Frozen ModernBERT.ipynb` (the best RoBERTa `<s>` settings from `hparam_search/`), so the encoder is the main change. Two things follow how Qwen3-Embedding is meant to be used:
1. **Last-token pooling.** Qwen3 is a decoder: with causal attention each token sees only the tokens before it, so only the last one has seen the whole message. The tokenizer appends `<|endoftext|>`, and the model was trained to put its summary there.
2. **A task instruction** in front of each message: `Instruct: Identify the topic or theme of the given news articles` + `\nQuery:`. This is the instruction Qwen used for 20 Newsgroups in its own MTEB evaluation.

**Predict before running:** with these settings, RoBERTa got 66.77 (LR) and 68.33 (MLP) on test, and ModernBERT's `[CLS]` 55.51 / 57.75. Where will Qwen land?

## Load dataset (copied from the PLAYGROUND)

In [1]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_20newsgroups

newsgroups_train = fetch_20newsgroups(
    subset='train', remove=('headers', 'footers', 'quotes'))

newsgroups_test = fetch_20newsgroups(
    subset='test', remove=('headers', 'footers', 'quotes'))

# Remove empty documents
for s, name in zip([newsgroups_train, newsgroups_test], ["train", "test"]):
  empty_indices = {i for i, doc in enumerate(s.data) if len(doc) == 0}
  orig_len = len(s.data)
  for k in ['data', 'filenames', 'target']:
    s[k] = [s[k][i] for i in range(orig_len) if i not in empty_indices]
  print(f"Removed {len(empty_indices)} empty documents from the {name} set. Before: {orig_len}. After: {len(s.data)}.")

y_train = np.array(newsgroups_train.target)
y_test = np.array(newsgroups_test.target)
n_classes = len(newsgroups_train.target_names)

Removed 218 empty documents from the train set. Before: 11314. After: 11096.
Removed 162 empty documents from the test set. Before: 7532. After: 7370.


## Qwen3-Embedding last-token embeddings

The same structure as the RoBERTa and ModernBERT notebooks, with three differences:
- **The instruction** goes in front of every message. It takes 17 tokens, so a message gets about 494 of the 512 tokens instead of 510.
- **Left padding:** the padding goes before short messages, so the last token of every message in a batch sits at position -1.
- **bfloat16**, the model's native precision (the vectors are converted to float32). Qwen's own examples also L2-normalize the vector, for cosine similarity; that's skipped here because the inputs are standardized below anyway.

The result is saved to `qwen3_0.6b_last_20ng.npz`, so later runs skip this cell's slow part (5–8 minutes on the GPU, depending on what else is running).

In [2]:
import os
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
CACHE = "qwen3_0.6b_last_20ng.npz"
INSTRUCTION = "Instruct: Identify the topic or theme of the given news articles\nQuery:"

## X is a list of strings (messages). Returns a np 2D array of shape (len(X), 1024):
## the last-token embedding of each message, with the task instruction in front
def qwen_embed(X, tokenizer, qwen, batch_size=32):
    vecs = []
    for i in tqdm(range(0, len(X), batch_size)):
        batch = tokenizer([INSTRUCTION + x for x in X[i:i + batch_size]], padding=True, truncation=True,
                          max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():  # frozen: no gradients needed
            out = qwen(**batch)
        # Padding is on the left, so position -1 is every message's last token (<|endoftext|>)
        vecs.append(out.last_hidden_state[:, -1].float().cpu().numpy())
    return np.concatenate(vecs)

if os.path.exists(CACHE):
    saved = np.load(CACHE)
    X_train, X_test = saved["X_train"], saved["X_test"]
else:
    # padding_side="left" so that every message's last token ends up at the same position, -1
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B", padding_side="left")
    # bfloat16 is the model's native precision and halves its memory
    qwen = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-0.6B", torch_dtype=torch.bfloat16).to(device).eval()
    X_train = qwen_embed(newsgroups_train.data, tokenizer, qwen)
    X_test = qwen_embed(newsgroups_test.data, tokenizer, qwen)
    np.savez(CACHE, X_train=X_train, X_test=X_test)

print(X_train.shape, X_test.shape)

/home/tonyxdsu/venvs/cs423/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|          | 0/347 [00:00<?, ?it/s]

  0%|          | 1/347 [00:01<11:10,  1.94s/it]

  1%|          | 2/347 [00:02<06:55,  1.20s/it]

  1%|          | 3/347 [00:03<05:29,  1.05it/s]

  1%|          | 4/347 [00:03<04:48,  1.19it/s]

... (558 outputs left out)

100%|█████████▉| 230/231 [03:19<00:00,  1.11it/s]

100%|██████████| 231/231 [03:19<00:00,  1.45it/s]

100%|██████████| 231/231 [03:19<00:00,  1.16it/s]

(11096, 1024) (7370, 1024)


## Baseline: logistic regression

First, hold out 10% of train as a dev set, the same split as the RoBERTa notebooks. The MLP needs it for early stopping; the test set is only used at the very end. LR trains on the same 90%, so the comparison stays fair.

LR uses the RoBERTa search's best settings:
- **Standardized inputs** (mean 0 and std 1 in every dimension, from the 90%'s statistics only). Qwen's values spread about 68× wider than RoBERTa's `<s>` values (median per-dimension std 1.74 vs. 0.025). On raw inputs, the same `C` would mean a different amount of regularization. Standardized, both encoders are on the same scale, so RoBERTa's best `C` is a fair guess.
- **`C = 0.003`**, the best for standardized RoBERTa features (70.78 CV, 66.77 test).
- **1 BLAS thread** (`threadpool_limits(1)`), which is about 60× faster on this machine.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from threadpoolctl import threadpool_limits

X_tr, X_dev, y_tr, y_dev = train_test_split(
    X_train, y_train, test_size=0.1, stratify=y_train, random_state=423)

scaler = StandardScaler().fit(X_tr)  # mean and std of the 90% only, so dev and test stay unseen
X_tr, X_dev = scaler.transform(X_tr), scaler.transform(X_dev)

with threadpool_limits(1):
    logreg = LogisticRegression(C=0.003, max_iter=1000).fit(X_tr, y_tr)
lr_acc = metrics.accuracy_score(y_test, logreg.predict(scaler.transform(X_test)))
print(f"LR test accuracy: {lr_acc*100:.2f}")

LR test accuracy: 72.90


## MLP

1024 → 2048 → 64 → 20, the best RoBERTa config from the search (with Qwen's 1,024-d input instead of 768). The search found that these mattered:
- **A small learning rate:** AdamW at lr 1e-4. The first notebook's 1e-3 is too big a step once inputs are standardized. Weight decay stays at AdamW's default, 0.01.
- **Heavy dropout in every place:** 0.2 on the input, 0.7 after hidden layer 1, 0.5 after hidden layer 2. Without dropout, the MLP was only level with LR.
- **Label smoothing 0.1:** the training target puts 90% of the probability on the right class and spreads the other 10% evenly over all classes, so the model is never pushed toward 100% confidence.

GELU and batch size 32 also come from the search, but barely mattered there.

In [4]:
torch.manual_seed(423)

mlp = nn.Sequential(
    nn.Dropout(0.2),                                    # dropout on the input embedding
    nn.Linear(1024, 2048), nn.GELU(), nn.Dropout(0.7),  # hidden layer 1: wide, heavy dropout
    nn.Linear(2048, 64), nn.GELU(), nn.Dropout(0.5),    # hidden layer 2: narrow
    nn.Linear(64, n_classes),                           # one logit per class
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(mlp.parameters(), lr=1e-4)  # weight decay: AdamW's default, 0.01

Training loop with early stopping, as in the A1 search: stop once dev loss hasn't improved for `PATIENCE` epochs, then go back to the best epoch's weights. Two changes from the first notebook, both from the search:
- **Patience 10 and up to 300 epochs** (was 5 and 100). A 10× smaller learning rate needs more epochs: at lr 1e-4, RoBERTa's best epoch was around 50–90.
- **Dev loss is plain cross-entropy, without label smoothing.** The train loss includes smoothing, so it can't go below about 0.6 and isn't directly comparable to the dev loss.

In [5]:
import copy
import time
from torch.utils.data import TensorDataset, DataLoader

loader = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)), batch_size=32, shuffle=True)
X_dev_t, y_dev_t = torch.tensor(X_dev).to(device), torch.tensor(y_dev).to(device)

PATIENCE = 10
best = {"dev_loss": float("inf"), "epoch": 0, "state": None}
start = time.perf_counter()

for epoch in range(1, 301):
    mlp.train()  # dropout on
    train_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)

    mlp.eval()  # dropout off
    with torch.no_grad():
        dev_logits = mlp(X_dev_t)
        dev_loss = nn.functional.cross_entropy(dev_logits, y_dev_t).item()  # no label smoothing, as in the search
        dev_acc = (dev_logits.argmax(dim=1) == y_dev_t).float().mean().item()
    print(f"epoch {epoch:3d}  train loss {train_loss / len(X_tr):.3f}  dev loss {dev_loss:.3f}  dev acc {dev_acc*100:.2f}")

    if dev_loss < best["dev_loss"]:
        best = {"dev_loss": dev_loss, "epoch": epoch, "state": copy.deepcopy(mlp.state_dict())}
    elif epoch - best["epoch"] >= PATIENCE:
        break

mlp.load_state_dict(best["state"])
print(f"Training took {time.perf_counter() - start:.1f} s. Using epoch {best['epoch']} (lowest dev loss)")

epoch   1  train loss 1.943  dev loss 0.994  dev acc 73.06


epoch   2  train loss 1.523  dev loss 0.910  dev acc 73.78


epoch   3  train loss 1.455  dev loss 0.877  dev acc 74.95


epoch   4  train loss 1.419  dev loss 0.848  dev acc 75.32


epoch   5  train loss 1.392  dev loss 0.826  dev acc 75.68


epoch   6  train loss 1.363  dev loss 0.820  dev acc 75.59


... (54 outputs left out)

epoch  61  train loss 1.071  dev loss 0.695  dev acc 79.37


epoch  62  train loss 1.063  dev loss 0.689  dev acc 79.19


epoch  63  train loss 1.060  dev loss 0.691  dev acc 79.64


epoch  64  train loss 1.065  dev loss 0.693  dev acc 79.19
Training took 228.8 s. Using epoch 54 (lowest dev loss)


## LR vs. MLP on the test set

In [6]:
mlp.eval()
with torch.no_grad():
    mlp_pred = mlp(torch.tensor(scaler.transform(X_test)).to(device)).argmax(dim=1).cpu().numpy()
mlp_acc = metrics.accuracy_score(y_test, mlp_pred)

print(f"LR  test accuracy: {lr_acc*100:.2f}")
print(f"MLP test accuracy: {mlp_acc*100:.2f}")
print(f"Difference: {(mlp_acc - lr_acc)*100:+.2f} points (1 point = {len(y_test) / 100:.0f} test messages)")

LR  test accuracy: 72.90
MLP test accuracy: 74.15
Difference: +1.25 points (1 point = 74 test messages)


For comparison, with the same settings (test accuracy):

| | LR | MLP |
|---|---|---|
| RoBERTa `<s>` (`hparam_search/final_test.json`; MLP: mean of 10 seeds) | 66.77 | 68.33 |
| ModernBERT `[CLS]` (`Frozen ModernBERT.ipynb`) | 55.51 | 57.75 |

This is one MLP with one seed, and the settings were tuned for RoBERTa, not Qwen. A different seed shifts the result a little, so read a gap under about a point as "about the same".